# 🚀 RL Trading — Main Notebook

Entry point cho training và inference.

## 1. Training

In [ ]:
from src.logging_config import setup_logging
from src.config import Config
from src.training.trainer import Trainer

# Đọc configs/logging.yaml: console WARNING+, file DEBUG chia nhỏ 5MB
log_dir = setup_logging()
print(f"Log dir: {log_dir}")

cfg = Config.load("configs/")
trainer = Trainer(cfg)
trainer.run()

print(f"\n✅ Output directory: {trainer.run_dir}")

Log dir: artifacts\logs\train_20260528_215557
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
[Split] Train=1028 | Val=129 | Test=129
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
[Split] Train=1028 | Val=129 | Test=129
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
[Split] Train=1028 | Val=129 | Test=129
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
[Split] Train=1028 | Val=129 | Test=129


Training:   0%|          | 0/500 [00:00<?, ?ep/s]

## 2. Evaluate (Test Set)

In [ ]:
# Evaluate trên test set với best model
test_results = trainer.evaluate()

# Export ROI table
trainer.export_roi_table(test_results)

## 3. Biểu đồ

In [ ]:
# Vẽ training curves + per-stock charts
# Tất cả output vào: outputs/output_<timestamp>/charts/
trainer.generate_charts(test_results)

print(f"Charts → {trainer.charts_dir}")

## 4. Inference (placeholder)

In [ ]:
from src.config import Config
from src.inference.inferencer import Inferencer

cfg = Config.load("configs/")
infer = Inferencer(cfg, model_path=str(trainer.weights_dir / "best_model.pkl"))

# Predict cho 1 stock
result = infer.predict("data/VNM.csv")
print(result)

## 5. Hyperparameter Sweep

Vòng lặp train từng bộ hyperparameter trọng tâm.  
Dùng `cfg.override()` để thay đổi giá trị trong memory — không sửa file config.

In [ ]:
# ════════════════════════════════════════════════════════════════
# Define experiments — mỗi dict là 1 run với overrides cụ thể
# Chỉ thay đổi 1 param (hoặc 1 cặp liên quan) mỗi lần
# ════════════════════════════════════════════════════════════════

EXPERIMENTS = [
    # ── Tier 1: gamma (discount factor) ─────────────────────────
    {"name": "gamma_090", "overrides": {"agent.gamma": 0.90}},
    {"name": "gamma_095", "overrides": {"agent.gamma": 0.95}},
    {"name": "gamma_097", "overrides": {"agent.gamma": 0.97}},
    {"name": "gamma_099", "overrides": {"agent.gamma": 0.99}},

    # ── Tier 1: learn_every (gradient updates frequency) ────────
    {"name": "learn_every_2", "overrides": {"training.learn_every": 2}},
    {"name": "learn_every_4", "overrides": {"training.learn_every": 4}},
    {"name": "learn_every_8", "overrides": {"training.learn_every": 8}},

    # ── Tier 1: lr + lr_decay (learning rate schedule) ──────────
    {"name": "lr_high",  "overrides": {"agent.lr": 0.001, "agent.lr_decay": 0.998}},
    {"name": "lr_mid",   "overrides": {"agent.lr": 0.0005, "agent.lr_decay": 0.995}},
    {"name": "lr_low",   "overrides": {"agent.lr": 0.0003, "agent.lr_decay": 0.999}},

    # ── Tier 2: batch_size + buffer_cap ─────────────────────────
    {"name": "batch128_buf50k", "overrides": {"agent.batch_size": 128, "agent.buffer_cap": 50000}},
    {"name": "batch256_buf20k", "overrides": {"agent.batch_size": 256, "agent.buffer_cap": 20000}},
    {"name": "batch256_buf50k", "overrides": {"agent.batch_size": 256, "agent.buffer_cap": 50000}},

    # ── Tier 2: tau (soft target update rate) ───────────────────
    {"name": "tau_001", "overrides": {"agent.tau": 0.001}},
    {"name": "tau_005", "overrides": {"agent.tau": 0.005}},
    {"name": "tau_01",  "overrides": {"agent.tau": 0.01}},

    # ── Tier 2: eps_decay (exploration schedule) ────────────────
    {"name": "eps_decay_990", "overrides": {"agent.eps_decay": 0.990}},
    {"name": "eps_decay_995", "overrides": {"agent.eps_decay": 0.995}},
    {"name": "eps_decay_997", "overrides": {"agent.eps_decay": 0.997}},
    {"name": "eps_decay_999", "overrides": {"agent.eps_decay": 0.999}},
]

print(f"Total experiments: {len(EXPERIMENTS)}")

Total experiments: 3


In [2]:
import json
from src.logging_config import setup_logging
from src.config import Config
from src.training.trainer import Trainer

setup_logging()

# ════════════════════════════════════════════════════════════════
# Sweep loop — train từng experiment
# ════════════════════════════════════════════════════════════════
results_summary = []

for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*60}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}] Experiment: {exp['name']}")
    print(f"  Overrides: {exp['overrides']}")
    print(f"{'='*60}")

    # Load fresh config mỗi lần (reset về baseline)
    cfg = Config.load("configs/")

    # Override hyperparameters
    for key, val in exp["overrides"].items():
        cfg.override(key, val)

    # Train
    try:
        trainer = Trainer(cfg)
        trainer.run()

        # Collect result từ logs.json
        with open(trainer.run_dir / "logs.json", encoding="utf-8") as f:
            run_log = json.load(f)

        results_summary.append({
            "experiment": exp["name"],
            "overrides": exp["overrides"],
            "run_dir": str(trainer.run_dir),
            "test_metrics": run_log["result"].get("test_metrics", {}),
            "best_score": run_log["result"]["best_score"],
            "elapsed_min": run_log["elapsed_minutes"],
        })
        print(f"  ✅ Done: score={run_log['result']['best_score']:.4f}")

    except Exception as e:
        print(f"  ❌ Failed: {e}")
        results_summary.append({
            "experiment": exp["name"],
            "overrides": exp["overrides"],
            "run_dir": "FAILED",
            "test_metrics": {},
            "best_score": 0,
            "elapsed_min": 0,
        })

print(f"\n\n{'='*60}")
print(f"  All {len(EXPERIMENTS)} experiments completed!")
print(f"{'='*60}")


  [1/3] Experiment: lr_high
  Overrides: {'agent.lr': 0.001, 'agent.lr_decay': 0.998}
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
[Split] Train=1028 | Val=129 | Test=129
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
[Split] Train=1028 | Val=129 | Test=129
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
[Split] Train=1028 | Val=129 | Test=129
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
[Split] Train=1028 | Val=129 | Test=129


Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260528_234158\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260528_234158\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260528_234158\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260528_234158\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260528_234158\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260528_234158\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260528_234158\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260528_234158\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260528_234158\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260528_234158\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260528_234158\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260528_234158\charts\FPT/0_da

Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_005309\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_005309\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_005309\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_005309\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_005309\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_005309\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_005309\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_005309\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_005309\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_005309\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_005309\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_005309\charts\FPT/0_da

Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_021133\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_021133\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_021133\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_021133\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_021133\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_021133\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_021133\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_021133\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_021133\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_021133\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_021133\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_021133\charts\FPT/0_da

In [3]:
import pandas as pd
import os

# ════════════════════════════════════════════════════════════════
# Summary table — so sánh tất cả experiments
# ════════════════════════════════════════════════════════════════
rows = []
for r in results_summary:
    avg = r["test_metrics"].get("average", {})
    rows.append({
        "Experiment": r["experiment"],
        "Return%": avg.get("return_pct", 0),
        "Sharpe": avg.get("sharpe", 0),
        "WinRate%": avg.get("win_rate", 0),
        "MaxDD%": avg.get("max_dd_pct", 0),
        "N_trades": avg.get("n_trades", 0),
        "PF": avg.get("profit_factor", 0),
        "Score": r["best_score"],
        "Time(min)": r["elapsed_min"],
    })

df_summary = pd.DataFrame(rows)

# Sort by Sharpe (chỉ số quan trọng nhất)
df_summary = df_summary.sort_values("Sharpe", ascending=False).reset_index(drop=True)

# Export CSV
os.makedirs("artifacts/outputs", exist_ok=True)
df_summary.to_csv("artifacts/outputs/hyperparam_sweep_results.csv", index=False)

print("\n📊 Hyperparameter Sweep Results (sorted by Sharpe):")
print("="*80)
display(df_summary)
print(f"\n💾 Saved → artifacts/outputs/hyperparam_sweep_results.csv")


📊 Hyperparameter Sweep Results (sorted by Sharpe):


,Experiment,Return%,Sharpe,WinRate%,MaxDD%,N_trades,PF,Score,Time(min)
0,lr_mid,-1.86,-0.5988,55.4,-6.05,16,1.82,0.3438,78.37
1,lr_high,-1.74,-0.6399,65.0,-11.01,16,3.66,0.3728,71.09
2,lr_low,-3.38,-1.0935,53.4,-8.80,15,1.14,0.3914,77.44



💾 Saved → artifacts/outputs/hyperparam_sweep_results.csv
